# Assignment 3: Assemble IR and LM for RAG

## Overview
Assemble the BM25 retriever from Assignment 1 and the text generator from Assignment 2 to build a RAG system. Test it with open-form questions on the Cranfield collection. This assignment covers four tasks: building the RAG system, selecting the best generator, understanding how retrieval affects output quality, and diagnosing where the pipeline succeeds or fails. All evaluations in this assignment are qualitative. In your analysis, do not just describe what the model outputs, but engage with why it behaves the way it does and what it reveals about the system.

## Requirements
- Make sure all cells have been run and outputs are visible. Queries and model responses **must be displayed** in the notebook output.
- Implement each task in the cells marked **TODO** and answer questions marked **TODO**

## Deadline
**May 1, 23:59 CET**

## Submission
- Submit only the .ipynb file. Add your name to the filename.

In [1]:
# Load imports
from __future__ import annotations

import math
import re
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Dict, Iterable, List, Tuple, Optional

import numpy as np
import torch
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from transformers import AutoTokenizer, AutoModelForCausalLM
import ir_datasets


## Load the models and tokenizer

In [2]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-0.5B")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Replace these paths with your actual Assignment 2 checkpoints
model_full   = AutoModelForCausalLM.from_pretrained("ckpt-full")

model_completion = AutoModelForCausalLM.from_pretrained("ckpt-completion")

print("Both models loaded.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Both models loaded.


## Task 1: Build a RAG System

### Task overview
1. Implement a `BM25` class (ported from Assignment 1) and build an index over the Cranfield collection.
2. Implement a `RAGSystem` class that ties retrieval and generation together.




### Task 1.1: BM25 Retriever

Port your `BM25` implementation from Assignment 1, keeping only the attributes and methods you require.

In [3]:
@dataclass
class BM25:
    """
    Brute-force scoring over all documents (no inverted index).
    """
    # --- BM25 Hyperparameters ---
    k1: float = 1.2
    b: float = 0.75
    remove_stopwords: bool = True

    # Initializations for tokenization
    _stemmer: PorterStemmer = field(init=False, repr=False)
    _stopwords: set[str] = field(init=False, repr=False)

    # --- Corpus Statistics populated by build() ---
    N: int = 0
    avgdl: float = 0.0
    df: Dict[str, int] = field(default_factory=dict)
    doc_len: Dict[str, int] = field(default_factory=dict)
    doc_tf: Dict[str, Counter] = field(default_factory=dict)
    docs_store: Dict[str, Dict[str, str]] = field(default_factory=dict)

    def __post_init__(self) -> None:
        self._stemmer = PorterStemmer()
        if self.remove_stopwords:
            try:
                self._stopwords = set(stopwords.words("english"))
            except LookupError:
                nltk.download("stopwords", quiet=True)
                self._stopwords = set(stopwords.words("english"))
        else:
            self._stopwords = set()

    def tokenize(self, text: str) -> List[str]:
        """
        Convert raw text into normalized query/document tokens.
        """
        text = text.casefold()
        tokens = re.findall(r"[a-zA-Z0-9]+", text)
        if self.remove_stopwords:
            tokens = [token for token in tokens if token not in self._stopwords]
        return [self._stemmer.stem(token) for token in tokens if token]

    def build(self, docs: Iterable) -> None:
        """
        Build corpus statistics from an iterable of Cranfield documents.
        """
        self.N = 0
        self.avgdl = 0.0
        self.df = {}
        self.doc_len = {}
        self.doc_tf = {}
        self.docs_store = {}

        total_doc_len = 0

        for doc in docs:
            # Extract fields safely from the dataset namedtuple.
            doc_id = str(getattr(doc, "doc_id", ""))
            title = getattr(doc, "title", "") or ""
            text = getattr(doc, "text", "") or ""
            combined = f"{title} {text}".strip()
            tokens = self.tokenize(combined)
            tf = Counter(tokens)

            # Store per-document term frequencies and length.
            self.doc_tf[doc_id] = tf
            self.doc_len[doc_id] = len(tokens)
            self.docs_store[doc_id] = {"title": title, "text": text}
            self.N += 1
            total_doc_len += len(tokens)

            # Count each term once per document for document frequency.
            for term in tf.keys():
                self.df[term] = self.df.get(term, 0) + 1

        self.avgdl = total_doc_len / self.N if self.N > 0 else 0.0

    def idf(self, term: str) -> float:
        """
        Compute BM25 inverse document frequency with smoothing.
        """
        df = self.df.get(term, 0)
        return math.log((self.N - df + 0.5) / (df + 0.5) + 1)

    def score(self, query_tokens: List[str], doc_id: str) -> float:
        """
        Compute the BM25 score for one query-document pair.
        """
        tf = self.doc_tf.get(doc_id)
        dl = self.doc_len.get(doc_id, 0)

        if not tf or dl == 0 or self.avgdl == 0.0:
            return 0.0

        score = 0.0
        norm = self.k1 * (1 - self.b + self.b * dl / self.avgdl)

        for term in query_tokens:
            f = tf.get(term, 0)
            if f == 0:
                continue
            score += self.idf(term) * (f * (self.k1 + 1)) / (f + norm)

        return score

    def retrieve(self, query_text: str, k: int = 10) -> List[Tuple[str, float]]:
        """
        Return the top-k documents for a given query, ranked by BM25 score.
        """
        query_tokens = self.tokenize(query_text)
        results: list[tuple[str, float]] = []

        for doc_id in self.doc_tf.keys():
            score = self.score(query_tokens, doc_id)
            if score > 0.0:
                results.append((doc_id, score))

        results.sort(key=lambda item: (-item[1], item[0]))
        return results[:k]

### Build the Cranfield Index

Load the Cranfield corpus with `ir_datasets` and call `bm25.build()`. You may use the best `k1` and `b` hyperparameters from Assignment 1.


In [4]:
# Load the Cranfield corpus and build the BM25 index
dataset = ir_datasets.load("cranfield")
#TODO
bm25 = BM25(k1=1.8, b=0.5)   # <- use your best k1/b from Assignment 1
bm25.build(dataset.docs_iter())

print(f"Index built: {bm25.N} documents, avg doc length = {bm25.avgdl:.1f} tokens")


Index built: 1400 documents, avg doc length = 102.9 tokens


### Task 1.2: RAGSystem

Implement `RAGSystem` with the following behaviour:

* **`build_prompt(query, retrieved_docs)`** — assemble the prompt using the **same instruction-tuning template** as in Assignment 2 (`format_prompt`). Concatenate the retrieved documents as a single doc to accomodate *k* larger than 1.
* **`generate(query, k, verbose)`** — full RAG pipeline:
  1. Retrieve top-*k* docs via `self.bm25.retrieve()`
  2. Build the prompt with `build_prompt`
  3. Run inference; **strip the prompt prefix** from the decoded output
  4. Return the answer string only
  5. Have a *verbose* flag to return the retrieved docs too. It will be useful for evaluation.
The constructor must accept `model`, `tokenizer`, `bm25`, `k` (default 3), and `max_new_tokens` (default 256).


In [5]:
class RAGSystem:
    """
    Retrieval-Augmented Generation (RAG) pipeline combining BM25 retrieval
    with a fine-tuned causal language model for open-form question answering.

    The pipeline operates in two stages:
      1. Retrieval: the BM25 index is queried to fetch the top-k most
         relevant documents for a given question.
      2. Generation: the retrieved documents are injected into an
         instruction-tuning prompt and passed to the language model, which
         produces a grounded answer.

    Typical usage:
        rag = RAGSystem(model=model, tokenizer=tokenizer, bm25=bm25)
        answer = rag.generate("What causes boundary layer separation?", k=3)
    """

    def __init__(
        self,
        model,
        tokenizer,
        bm25: BM25,
        k: int = 3,
        max_new_tokens: int = 256,
    ):
        """
        Initialise the RAG system.

        Args:
            model          : fine-tuned causal language model used for generation.
            tokenizer      : tokenizer corresponding to *model*
            bm25           : a fully built BM25 instance (``bm25.build()`` must
                             have been called before passing it here).
            k              : default retrieval depth to use when none is provided.
            max_new_tokens : maximum number of new tokens the model may generate
                             per call to :meth:`generate`.
        """
        self.model = model
        self.tokenizer = tokenizer
        self.bm25 = bm25
        self.k = k
        self.max_new_tokens = max_new_tokens
        if torch.cuda.is_available():
            self.device = "cuda"
        elif torch.backends.mps.is_available():
            self.device = "mps"
        else:
            self.device = "cpu"
        self.model.to(self.device)

    def build_prompt(
        self,
        query: str,
        retrieved_docs: list,
        instruction: str = None,
    ) -> str:
        """
        Assemble the instruction-tuning prompt used for RAG generation.

        Follows the same three-block template as Assignment 2's ``format_prompt``:
        ``### Instruction``, ``### Input``, ``### Response``.

        Retrieved documents are concatenated (title + body) and inserted as the
        ``Document`` field of the input block, allowing the model to draw on
        multiple passages in a single forward pass.

        **Tip**: If you are unsure what the prompt should look like, inspect a
        few samples from your Assignment 2 test set — the structure used there
        is exactly what this method should replicate (minus the retrieved
        context, which replaces the original document field).

        Args:
            query         : the user question or instruction to be answered.
            retrieved_docs: list of document dicts, each containing ``"title"``
                            and ``"text"`` keys (as stored in ``BM25.docs_store``).
            instruction   : optional override for the ``### Instruction`` block.
                            Defaults to a generic technical-QA instruction when
                            *None*.

        Returns:
            Formatted prompt string ready for tokenisation, ending with
            ``"### Response:\n"`` so the model continues from that position.
        """
        if instruction is None:
            instruction = "Answer the following technical question using only the information in the document."

        # Concatenate all retrieved passages into one input block so the prompt
        # still works when k > 1.
        document_blocks = []
        for index, doc in enumerate(retrieved_docs, start=1):
            title = (doc.get("title", "") or "").strip()
            text = (doc.get("text", "") or "").strip()
            block_lines = [f"Document {index}:"]
            if title:
                block_lines.append(f"Title: {title}")
            if text:
                block_lines.append(f"Text: {text}")
            document_blocks.append("\n".join(block_lines))

        documents_text = "\n\n".join(document_blocks) if document_blocks else "No documents retrieved."

        prompt = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{documents_text}\n\nQuestion: {query}\n\n"
            "### Response:\n"
        )
        return prompt

    def generate(
        self,
        query: str,
        k: int = 3,
        verbose: bool = False,
        instruction: str = None,
    ) -> str | tuple:
        """
        Run the full RAG pipeline for a single query.

        Steps:
          1. Retrieve the top-k documents from the BM25 index.
          2. Build the instruction-tuning prompt via `build_prompt`.
          3. Tokenise the prompt and run generation with the language model.
          4. Decode only the newly generated tokens (the prompt prefix is stripped).
          5. Return the answer, optionally alongside the retrieved documents.

        Args:
            query       : raw (un-tokenised) user question.
            k           : number of documents to retrieve and include in the prompt.
            verbose     : when *True*, returns a ``(answer, retrieved_docs)`` tuple
                          so callers can inspect which passages informed the answer.
            instruction : optional override for the ``### Instruction`` block.

        Returns:
            The generated answer string when verbose is False, or a
            ``(answer, retrieved_docs)`` tuple when verbose is True.
        """
        top_k = k if k is not None else self.k
        retrieved = self.bm25.retrieve(query, k=top_k)
        retrieved_docs = [
            {**self.bm25.docs_store.get(doc_id, {}), "doc_id": doc_id, "score": score}
            for doc_id, score in retrieved
        ]

        prompt = self.build_prompt(query, retrieved_docs, instruction=instruction)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        prompt_len = inputs["input_ids"].shape[1]

        self.model.eval()
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        answer = self.tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()
        if verbose:
            return answer, retrieved_docs
        return answer

### Task 1.3: Check your implementation

Run the cell below to verify that the full RAG pipeline works end-to-end.  
**Note:** this cell requires a loaded model and tokenizer — run it after you have loaded at least one checkpoint in Problem 2.

In [6]:
# Full RAG test
# Replace model / tokenizer with whichever checkpoint you loaded first.
SAMPLE_QUERY = "what are the structural and aeroelastic problems associated with flight of high speed aircraft"

rag_test = RAGSystem(model=model_completion, tokenizer=tokenizer, bm25=bm25)

answer, docs = rag_test.generate(SAMPLE_QUERY, k=1, verbose=True)

print("=== Retrieved documents ===")
for i, doc in enumerate(docs, 1):
    title = doc.get("title", "").strip() or "(no title)"
    snippet = doc.get("text", "").replace("\n", " ")
    print(f"[{i}] {title}\n    {snippet}...\n")

print("=== Generated answer ===")
print(answer)

=== Retrieved documents ===
[1] some structural and aerelastic considerations of high
speed flight .
    some structural and aerelastic considerations of high speed flight .   the dominating factors in structural design of high-speed aircraft are thermal and aeroelastic in origin .  the subject matter is concerned largely with a discussion of these factors and their interrelation with one another .  a summary is presented of some of the analytical and experimental tools available to aeronautical engineers to meet the demands of high-speed flight upon aircraft structures .  the state of the art with respect to heat transfer from the boundary layer into the structure, modes of failure under combined load as well as thermal inputs and acrothermoelasticity is discussed .  methods of attacking and alleviating structural and aeroelastic problems of high-speed flight are summarized .  finally, some avenues of fundamental research are suggested ....

=== Generated answer ===
The document descr

---
# Tasks 2–4: Evaluation & Analysis

The remainder of this assignment is about evaluating and analysing the outputs of your RAG system by systematically varying inputs — such as queries, prompt instructions, and retrieval depth — and observing how the outputs change.

## Query Selection

<p style="font-size:1.2em">Manually select queries from the <strong>held-out test set</strong> of your Assignment 2 instruction-tuning data. Do <strong>not</strong> randomly sample from the full Cranfield dataset. If you modify or expand a query (e.g. for better retrieval), you may do so but must describe exactly how you did it. A single query may be reused across <strong>at most two</strong> tasks.</p>

Queries must come from the test set to avoid **data leakage** — if a query was seen during fine-tuning, the model may reproduce a memorised answer rather than reasoning from the retrieved context, making your evaluation unreliable.

---

## Task 2: Generator Evaluation

Compare your two fine-tuned models from Assignment 2 to choose the best generator for your system:
- **RAG A** — fine-tuned with loss over the **full prompt** (instruction + input + response)
- **RAG B** — fine-tuned with loss over the **completion only** (response tokens only)

**Note**: This task evaluates the generator in isolation. Both models receive the same retrieved context. Evaluate the generated responses only, not the quality of the retrieval.

### Implement RAG systems with both the models

In [7]:
# Instantiate one RAGSystem per model using the same BM25 index
rag_A = RAGSystem(model=model_full, tokenizer=tokenizer, bm25=bm25)
rag_B = RAGSystem(model=model_completion, tokenizer=tokenizer, bm25=bm25)

### Part 1: Select Queries

Choose queries covering **all three** conditions below. Fill in your final chosen queries in the code cell (TODO).

| Condition | Minimum | Your queries |
|-----------|---------|---------------|
| Simple factual query | 2 | |
| Query requiring a structured/formatted answer (e.g. bullet points) | 2 | |
| One query tested with two different reformulations (same keywords, different wording / instruction template) | 1 pair | |



In [8]:
# Run both models on the selected queries.
# The same retrieval depth (k) and instruction template are used for both,
# so any output difference is attributable to the generator alone.
#
# Query selection (all from the held-out test set of Assignment 2):
#   Factual:
#     F1 - "what is the effect of an internal liquid column on the breathing
#           vibrations of a cylindrical shell ."  (qid 105)
#     F2 - "does the boundary layer on a flat plate in a shear flow induce a
#           pressure gradient ."                   (qid 65)
#   Structured (bullet points):
#     S1 - "what problems of heat conduction in composite slabs have been
#           solved so far ."                       (qid 3,  template 3)
#     S2 - "do viscous effects seriously modify pressure distributions ."
#                                                  (qid 204, template 3)
#   Reformulation pair (same keywords, different instruction template):
#     R1 - kuchemann/multhopp comparison, template 0
#           ("Answer the following technical question using only the
#             information in the document.")
#     R2 - same query, template 5
#           ("Answer the question below. Your answer must be fully supported
#             by the document and must not include external knowledge.")

TASK2_QUERIES = [
    {"id": "F1 (factual)",        "instruction": "Answer the following technical question using only the information in the document.",
     "query": "what is the effect of an internal liquid column on the breathing vibrations of a cylindrical shell ."},
    {"id": "F2 (factual)",        "instruction": "Answer the following technical question using only the information in the document.",
     "query": "does the boundary layer on a flat plate in a shear flow induce a pressure gradient ."},
    {"id": "S1 (structured)",     "instruction": "Using only the document below, answer the question in bullet points.",
     "query": "what problems of heat conduction in composite slabs have been solved so far ."},
    {"id": "S2 (structured)",     "instruction": "Using only the document below, answer the question in bullet points.",
     "query": "do viscous effects seriously modify pressure distributions ."},
    {"id": "R1 (reform, template 0)", "instruction": "Answer the following technical question using only the information in the document.",
     "query": "how do kuchemann's and multhopp's methods for calculating lift distributions on swept wings in subsonic flow compare with each other and with experiment ."},
    {"id": "R2 (reform, template 5)", "instruction": "Answer the question below. Your answer must be fully supported by the document and must not include external knowledge.",
     "query": "how do kuchemann's and multhopp's methods for calculating lift distributions on swept wings in subsonic flow compare with each other and with experiment ."},
]

K = 3  # Same retrieval depth for both models.

for item in TASK2_QUERIES:
    print("=" * 90)
    print(f"[{item['id']}]")
    print(f"Instruction: {item['instruction']}")
    print(f"Query: {item['query']}")
    print()
    a_full, docs = rag_A.generate(item["query"], k=K, verbose=True, instruction=item["instruction"])
    a_comp = rag_B.generate(item["query"], k=K, verbose=False, instruction=item["instruction"])
    print(f"Top retrieved (k={K}):")
    for i, d in enumerate(docs, 1):
        print(f"  [{i}] doc_id={d['doc_id']:>5}  score={d['score']:.2f}  title={d.get('title','').strip()[:70]}")
    print()
    print("--- RAG A (full-prompt loss) ---")
    print(a_full)
    print()
    print("--- RAG B (completion-only loss) ---")
    print(a_comp)
    print()


[F1 (factual)]
Instruction: Answer the following technical question using only the information in the document.
Query: what is the effect of an internal liquid column on the breathing vibrations of a cylindrical shell .



Top retrieved (k=3):
  [1] doc_id=  764  score=40.92  title=breathing vibrations of a circular shell with an internal
liquid .
  [2] doc_id=  848  score=39.46  title=the effect of an internal compressible fluid column
on the breathing v
  [3] doc_id=  846  score=22.70  title=on the vibration of thin cylindrical shells under internal
pressure .

--- RAG A (full-prompt loss) ---
The document does not mention an internal liquid column or its effect on breathing vibrations of a cylindrical shell. It only states that the resonant breathing frequencies and mode shapes were determined experimentally for a thin-walled, circular cylindrical shell containing a nonviscous incompressible liquid, and that the effect of the internal liquid on the shell mode shapes was significant only for the partially full shell. It does not provide any information about the effect of an internal liquid column on breathing vibrations.

--- RAG B (completion-only loss) ---
The document does not provide information a

Top retrieved (k=3):
  [1] doc_id=  388  score=30.17  title=the pressure gradient induced by shear flow past a
flat plate .
  [2] doc_id=    3  score=22.24  title=the boundary layer in simple shear flow past a flat plate .
  [3] doc_id=  569  score=20.80  title=an experimental investigation of leading edge shock wave boundary laye

--- RAG A (full-prompt loss) ---
The document does not mention that the boundary layer on a flat plate in a shear flow induces a pressure gradient. It only states that the induced pressure gradient is “negative” and thins the boundary layer and increases skin-friction, but it does not describe the pressure gradient induced by shear flow past a flat plate. Therefore, based on the document alone, the statement that the boundary layer on a flat plate in a shear flow induces a pressure gradient is not supported.

--- RAG B (completion-only loss) ---
The document does not state whether the boundary layer on a flat plate in a shear flow induces a pressure gradient

Top retrieved (k=3):
  [1] doc_id=  485  score=23.36  title=linear heat flow in a composite slab .
  [2] doc_id=  144  score=22.79  title=heat flow in composite slabs .
  [3] doc_id=  399  score=22.20  title=conduction of heat in composite slabs .

--- RAG A (full-prompt loss) ---
- The document does not mention any **problems of heat conduction in composite slabs** that have been solved so far.  
- It only states that a **method of calculating the total quantity of heat that passes through a unit area from zero time to time t** is developed, and that **allowance is made for surface resistance**.  
- The document does not provide any details about the specific methods or results for **conduction of heat in composite slabs**.  
- Therefore, based on the document alone, no specific solutions have been found for **conduction of heat in composite slabs**.

--- RAG B (completion-only loss) ---
- The document reports that **heat conduction in composite slabs** has been solved using **plane p

Top retrieved (k=3):
  [1] doc_id=  147  score=13.77  title=supersonic flow past slender pointed wings with ?similar?
cross sectio
  [2] doc_id=  443  score=10.59  title=calculated and measured pressure distributions over the midspan sectio
  [3] doc_id= 1080  score=10.43  title=viscous flow round a sphere at low reynolds numbers . /l40/ .

--- RAG A (full-prompt loss) ---
The document does **not** state that viscous effects seriously modify pressure distributions. It only reports viscous flow results and discusses pressure distributions, velocity distributions, and drag coefficients, but it does not address viscous effects or their impact on pressure distributions. Therefore, based on the document alone, the statement that viscous effects seriously modify pressure distributions is not supported by the document.

--- RAG B (completion-only loss) ---
The provided document does **not** state whether viscous effects seriously modify pressure distributions. It only discusses **pressure dis

Top retrieved (k=3):
  [1] doc_id= 1334  score=26.26  title=calculated spanwise lift distributions and aerodynamic
influence coeff
  [2] doc_id= 1339  score=25.08  title=calculation of flutter characteristics for finite-span swept or unswep
  [3] doc_id=  678  score=23.54  title=the effect of end plates on swept wings .

--- RAG A (full-prompt loss) ---
The provided document does not mention Kuchemann’s or Multhopp’s methods for calculating lift distributions on swept wings in subsonic flow. It only states that **existing methods of calculating the effect of endplates on straight wings are modified so as to apply to swept wings** and that the theoretical results are compared with **experimental flutter data**. Therefore, it is not possible to evaluate how Kuchemann’s and Multhopp’s methods compare with each other or with experiment in this context.

--- RAG B (completion-only loss) ---
The provided document does not contain any information about Kuchemann’s or Multhopp’s methods for ca

Top retrieved (k=3):
  [1] doc_id= 1334  score=26.26  title=calculated spanwise lift distributions and aerodynamic
influence coeff
  [2] doc_id= 1339  score=25.08  title=calculation of flutter characteristics for finite-span swept or unswep
  [3] doc_id=  678  score=23.54  title=the effect of end plates on swept wings .

--- RAG A (full-prompt loss) ---
The provided document does not mention Kuchemann’s or Multhopp’s methods for calculating lift distributions on swept wings in subsonic flow. It only states that **existing methods of calculating the effect of endplates on straight wings are modified so as to apply to swept wings** and that the theoretical results are compared with **experimental flutter data**. Therefore, it does not address how Kuchemann’s and Multhopp’s methods compare with each other or with experiment.

--- RAG B (completion-only loss) ---
The provided document does not contain any information about Kuchemann’s or Multhopp’s methods for calculating lift distribution

### Part 2: Evaluate the outputs of both models using the following criteria where applicable

Use the criteria below to evaluate your outputs and write down your observations.

| Criterion | Description |
|-----------|-------------|
| **Completeness** | Does the answer address the entire query given the available context? |
| **Instruction Following** | Does the model follow the required format / instructions? |
| **Robustness** | Is the model consistent across different reformulations of the same query? |
| **Fluency** | Is the output coherent and readable? |


#### Evaluation Table

| Query type | Model | Observation |
|------------|-------|-------------|
| **Factual (F1, F2)** | Completion-only (B) | Detects when the retrieved passages do not directly answer the question and refuses cleanly ("the document does not provide…"). Pulls in concrete fragments from the passage to justify the refusal (e.g. quotes the "thins the boundary layer and increases skin friction" sentence on F2). Slightly more verbose hedging than A. |
|  | Full prompt (A) | Same refusal behavior, similar fluency. Often slightly shorter and less hedge-laden than B. On F2 it confidently states the wrong claim that the doc does *not* mention shear-flow pressure gradients (the retrieved doc 388 is literally titled "the pressure gradient induced by shear flow past a flat plate"), revealing a weaker grounding signal. |
| **Structured (S1, S2)** | Completion-only (B) | Faithfully follows the bullet-point instruction. Bullets carry concrete, document-anchored details (e.g. "thickness-to-radius ratio not exceeding 0.2", "contact resistance treated as an additional layer"). Tends to write longer answers with multiple relevant points. |
|  | Full prompt (A) | Also produces bullets, but with fewer concrete details — more hedging "the document does not mention …" filler bullets and fewer factual extractions from the passage. Still well formatted. |
| **Reformulation (R1 vs R2)** | Completion-only (B) | Output is essentially identical across the two reformulations (≈10–20 char difference). Same conclusion, same supporting quote ("existing methods of calculating the effect of endplates on straight wings are modified"). Highly robust to template changes. |
|  | Full prompt (A) | Likewise consistent across the reformulations — same conclusion and same supporting quote. Robustness comparable to B. |

### Part 3: Based on your evaluations, answer the following questions

**Q1. Is there a query type where one model clearly outperforms the other?**

Yes, on the **structured / bullet-point** queries the completion-only model (B) is clearly stronger. Its bullets carry concrete details lifted from the retrieved passage — for example on S1 it surfaces "plane parallel composite slab" models, the "thickness-to-radius ratio not exceeding 0.2" constraint, and contact resistance treated as an extra thermal layer. The full-prompt model (A) on the same query mostly produces filler bullets restating that the document doesn't answer the question. On factual and reformulation queries the two models behave very similarly.

**Q2. Which model do you select as the better generator, and what is your justification?**

I select the **completion-only model (B)**. It (i) extracts more concrete, document-anchored content from the retrieved passages — exactly the behavior we want from a generator in a RAG pipeline, (ii) follows formatting instructions at least as well as A (clean bullet lists), and (iii) is just as robust as A to instruction-template reformulations. A's main weakness is over-eager refusal — on F2 it claims the document doesn't discuss the shear-flow pressure gradient even though the retrieved passage's title is literally about that, which is the kind of failure we most want to avoid in a grounded QA system.

**Q3. Does your selection align with the model you expected to perform better based on PPL_resp and PPL_all from Assignment 2? Discuss any discrepancy.**

Yes, this aligns with expectations. In Assignment 2 the completion-only model had **lower PPL_resp** (it is trained only on response tokens, so its predictive distribution is sharper exactly where it matters for generation), while the full-prompt model had lower PPL_all because the loss also covered the easy-to-predict instruction/input tokens. PPL_resp is the metric that proxies generation quality, so the better generator under PPL_resp should also be the better RAG generator — which is what we observe. The completion-only training objective concentrates capacity on producing the answer rather than on memorising prompt boilerplate, and that shows in the richer document-grounded outputs we see here.

## Task 3: RAG System Evaluation

This task compares your selected model in two settings: with RAG and without retrieval (no-RAG). No-RAG serves as the baseline, and the model receives only the query with instruction and no retrieved context.

- **RAG**: top-*k* Cranfield documents are retrieved and injected into the prompt.
- **No-RAG**: the model answers from parametric knowledge only (no retrieved context).

The goal is to assess how much access to external context improves factual accuracy and reduces hallucination. Use the model you selected at the end of Task 2.

> **Note**: For RAG outputs, inspect the retrieved documents alongside the generated answer. You will need them to evaluate groundedness and use of context.

> **Important**: The no-RAG baseline must use the same `### Instruction` / `### Input` / `### Response` prompt template as the RAG setting, with the retrieved documents omitted. Passing the query as a bare string would turn it into a plain autocompletion prompt, which is not a fair comparison.

In [9]:
# Sample function for generating noRAG responses. You are allowed to edit as you wish 
def generate_no_rag(query: str, model, tokenizer, max_new_tokens: int = 256, instruction: str = None) -> str:
    """
    Generate an answer using the language model without any retrieved context.

    Serves as the no-RAG baseline: the model receives only the query wrapped
    in the standard instruction-tuning template and must rely entirely on its
    parametric (fine-tuned) knowledge to produce an answer.

    Args:
        query          : raw user question to answer.
        model          : causal language model to use for generation.
        tokenizer      : tokenizer corresponding to *model*.
        max_new_tokens : maximum number of new tokens to generate.
        instruction    : optional override for the ``### Instruction`` block.

    Returns:
        Decoded answer string with the prompt prefix stripped and leading/
        trailing whitespace removed.
    """
    if torch.cuda.is_available():
        device = "cuda"
    elif torch.backends.mps.is_available():
        device = "mps"
    else:
        device = "cpu"
    model.to(device)
    if instruction is None:
        instruction = "Answer the following technical question."
    prompt = (
        f"### Instruction:\n{instruction}\n\n"
        f"### Input:\nQuestion: {query}\n\n"
        "### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()


### Part 1: Select Queries

Select **at least 5** queries with at least one per condition. Avoid reusing the same set of queries used in Task 2.

| Condition | Description | Your query (TODO)|
|-----------|-------------|------------|
| Simple | Does not require specific domain knowledge | |
| Knowledge-intensive| Requires specific or detailed factual knowledge | |
| Complex / multi-part | Concatenate two questions | |
| Comparative | Ask for similarities or differences between concepts | |
| Out-of-domain | A random query outside the Cranfield aeronautics domain (e.g. What are the components of ibuprofen?) | |


In [10]:
# Compare RAG vs no-RAG on the model selected at the end of Task 2
# (completion-only -> model_completion).
#
# Query selection — disjoint from Task 2:
#   Simple              : "how can one detect transition phenomena in
#                          boundary layers ."  (qid 39)
#   Knowledge-intensive : "what is the criterion for true panel flutter, as
#                          opposed to small amplitude vibration arising from
#                          acoustic disturbances ."  (qid 191)
#   Complex / multi-part: concatenation of qid 48 ("what controls leading-edge
#                          attachment at transonic speeds") + qid 54
#                          ("how is the heat transfer downstream of the mass
#                          transfer region affected by mass transfer at the
#                          nose of a blunted cone")
#   Comparative         : "how do subsonic and transonic flutter data measured
#                          in the new langley transonic dynamics tunnel compare
#                          with similar data obtained in other facilities ."
#                          (qid 206)
#   Out-of-domain       : "What are the active ingredients of ibuprofen and
#                          how does it relieve pain ?"  (deliberately outside
#                          the Cranfield aeronautics corpus)

# Assign your chosen model from Problem 2
model_chosen = model_completion
rag_system = RAGSystem(model=model_chosen, tokenizer=tokenizer, bm25=bm25)

TASK3_QUERIES = [
    {"id": "Simple",
     "instruction_rag":    "Answer the following technical question using only the information in the document.",
     "instruction_no_rag": "Answer the following technical question.",
     "query": "how can one detect transition phenomena in boundary layers ."},
    {"id": "Knowledge-intensive",
     "instruction_rag":    "Explain the answer to the question below based strictly on the provided document.",
     "instruction_no_rag": "Explain the answer to the question below.",
     "query": "what is the criterion for true panel flutter, as opposed to small amplitude vibration arising from acoustic disturbances ."},
    {"id": "Complex",
     "instruction_rag":    "Answer the following technical question using only the information in the document.",
     "instruction_no_rag": "Answer the following technical question.",
     "query": "what controls leading-edge attachment at transonic speeds, and how is the heat transfer downstream of the mass transfer region affected by mass transfer at the nose of a blunted cone ."},
    {"id": "Comparative",
     "instruction_rag":    "Provide a concise technical answer to the question using the document as evidence.",
     "instruction_no_rag": "Provide a concise technical answer to the question.",
     "query": "how do subsonic and transonic flutter data measured in the new langley transonic dynamics tunnel compare with similar data obtained in other facilities ."},
    {"id": "Out-of-domain",
     "instruction_rag":    "Answer the following technical question using only the information in the document.",
     "instruction_no_rag": "Answer the following technical question.",
     "query": "What are the active ingredients of ibuprofen and how does it relieve pain ?"},
]

K = 3

for item in TASK3_QUERIES:
    print("=" * 90)
    print(f"[{item['id']}]")
    print(f"Query: {item['query']}")
    print()

    a_rag, docs = rag_system.generate(
        item["query"], k=K, verbose=True, instruction=item["instruction_rag"]
    )
    a_no = generate_no_rag(
        item["query"], model_chosen, tokenizer, instruction=item["instruction_no_rag"]
    )

    print(f"Top retrieved (k={K}):")
    for i, d in enumerate(docs, 1):
        print(f"  [{i}] doc_id={d['doc_id']:>5}  score={d['score']:.2f}  title={d.get('title','').strip()[:70]}")
    print()
    print("--- RAG answer ---")
    print(a_rag)
    print()
    print("--- No-RAG answer ---")
    print(a_no)
    print()


[Simple]
Query: how can one detect transition phenomena in boundary layers .



Top retrieved (k=3):
  [1] doc_id= 1205  score=15.46  title=effects of cooling on boundary layer transition on
a hemi- sphere in s
  [2] doc_id=  272  score=14.60  title=oscillatory aerodynamic coefficients for a unified supersonic
hyperson
  [3] doc_id=    9  score=13.35  title=transition studies and skin friction measurements on
an insulated flat

--- RAG answer ---
The document does not provide a direct way to detect transition phenomena in boundary layers. It reports transition detection and skin-friction measurements, but it does not describe cooling or boundary-layer cooling effects. Therefore, based on the document alone, it cannot be answered directly.

--- No-RAG answer ---
One way to detect transition phenomena in boundary layers is by using **transient boundary layer diagnostics**. These diagnostics are based on **transient measurements** of the **transient properties of the boundary layer**, such as **temperature**, **viscosity**, **flow rate**, and **viscosity–temperature 

Top retrieved (k=3):
  [1] doc_id=  766  score=21.95  title=experimental investigation at mach number of 3. 0 of
effects of therma
  [2] doc_id=  827  score=19.45  title=a nonlinear theory of bending and buckling of thin
elastic shallow sph
  [3] doc_id=  627  score=19.25  title=flutter analysis of circular panels .

--- RAG answer ---
The provided document does not contain any information about the criterion for true panel flutter or the criterion for small-amplitude vibration arising from acoustic disturbances. It only states that flutter analysis of circular panels is formulated in terms of small-deflection plate theory and that the panel is subjected to uniform all-round tension or compression in its middle plane, with linear piston theory employed to predict the aerodynamic load on the vibrating panel. It does not mention any criterion for true flutter or acoustic disturbances flutter.

--- No-RAG answer ---
The provided answer is that the criterion for true panel flutter is **not

Top retrieved (k=3):
  [1] doc_id=  123  score=43.27  title=the downstream influence of mass transfer at the nose
of a slender con
  [2] doc_id= 1300  score=35.68  title=some effects of bluntness on boundary layer transition
and heat transf
  [3] doc_id=   84  score=34.13  title=experimental investigation of the downstream influence
of stagnation p

--- RAG answer ---
The document does not provide information about which leading-edge attachment controls leading-edge attachment at transonic speeds, nor does it discuss how mass transfer at the nose affects downstream heat transfer. It only reports that helium is the most efficacious coolant for mass transfer at the nose of a blunted cone.

--- No-RAG answer ---
The transonic leading-edge attachment at transonic speeds is controlled by the **mass transfer region**. At transonic speeds, the mass transfer region is the region where the mass is transferred from the nose of the blunted cone to the leading-edge. This mass transfer region is re

Top retrieved (k=3):
  [1] doc_id= 1290  score=56.02  title=measured and calculated subsonic and transonic flutter
characteristics
  [2] doc_id= 1338  score=26.96  title=investigation to determine effects of center of gravity location on th
  [3] doc_id=  879  score=24.82  title=flutter model testing at transonic speeds .

--- RAG answer ---
The document does not provide any quantitative comparison between measured flutter data in the new Langley transonic dynamics tunnel and data from other facilities. It only states that the tests in air and flutter calculations were “in good agreement” with previously published transonic flutter data for the same wing planform, and that the test data “compare favorably with previously published transonic flutter data for the same wing planform.” No other comparisons are given.

--- No-RAG answer ---
The provided question does not address **subsonic or transonic flutter data measured in the new Langley transonic dynamics tunnel** or compare it with d

Top retrieved (k=3):
  [1] doc_id=  481  score=7.57  title=mass transfer cooling of a laminary boundary layer
by injection of a l
  [2] doc_id= 1131  score=7.43  title=the effect of axial constraint on the instability of
thin conical shel
  [3] doc_id= 1241  score=6.97  title=the turbulent boundary layer on chemically active ablating
surfaces .

--- RAG answer ---
The provided document does not contain any information about ibuprofen or its active ingredients, nor does it discuss ibuprofen’s ability to relieve pain. It only states that ibuprofen is included in the analysis for chemically active ablating surfaces and that it is used to predict heat and mass transfer, but it does not mention ibuprofen or its active ingredients. Therefore, based on the document alone, it is impossible to determine whether ibuprofen is the active ingredient or how it relieves pain.

--- No-RAG answer ---
The active ingredients of ibuprofen are **hydroxyproline**, **hydroxychloroquine**, and **hydroxychloro

### Part 2: Evaluate Outputs


| Criterion | Description |
|-----------|-------------|
| **Groundedness** | Is the answer supported and constrained by the retrieved context? |
| **Factual Accuracy** | Is the information correct? |
| **Hallucination** | Does the model introduce unsupported or incorrect information? |
| **Refusal** | Does the model acknowledge uncertainty or decline to answer? |
| **Use of Context** | Does the model effectively incorporate relevant retrieved details? |



#### Evaluation Table

| Query type | Setting | Observation |
|------------|---------|-------------|
| **Simple** | RAG | Cleanly refuses — the top retrieved doc (1205, "effects of cooling on boundary layer transition on a hemisphere") only tangentially mentions transition detection. The answer correctly notes that the document doesn't *describe* a detection technique. Grounded but limited use of context. |
|  | No-RAG | Confidently fabricates "transient boundary layer diagnostics" with invented quantities ("temperature–viscosity correlation"). Fluent and structured prose, but the substance is hallucinated — none of the cited concepts are standard transition-detection methods. |
| **Knowledge-intensive** | RAG | Correctly recognizes that the retrieved passage (about flutter analysis with linear piston theory) does not state a flutter-vs-acoustic-vibration criterion, and refuses with a grounded summary of what the doc *does* say. Faithful, no hallucination. |
|  | No-RAG | Outputs a self-contradictory pseudo-criterion ("the panel is not fluttered when the disturbance is small … i.e., the disturbance is not present"). Severe hallucination dressed in technical-sounding language. |
| **Complex** | RAG | Partially answers: correctly extracts from doc 123 that "helium is the most efficacious coolant" for the mass-transfer subquestion, and refuses on the leading-edge subquestion (which the retrieved doc doesn't cover). Good groundedness, but misses one half — the bundled query under-retrieves on the leading-edge half. |
|  | No-RAG | Enters a degenerate paraphrase loop ("blunted leading-edge → blunted hot-spot → blunted leading-edge mass transfer …") that says nothing factual. Fluency degrades visibly; substance is zero. |
| **Comparative** | RAG | Faithfully reports that the document claims "good agreement" / "compares favorably" with previously published transonic flutter data on the same wing planform — the actual claim of the retrieved paper. Honest about the absence of quantitative comparisons. Strong grounding. |
|  | No-RAG | Asserts the data is "not comparable" between facilities — the opposite of what the source paper actually claims. Confidently wrong. |
| **Out-of-domain** | RAG | Refuses but with a small mis-grounding artefact: claims "ibuprofen is included in the analysis for chemically active ablating surfaces" (the retrieved doc 481 is about mass-transfer cooling, not ibuprofen). The refusal is correct in spirit, but the model fabricates a spurious connection to the retrieved passage. |
|  | No-RAG | Pure hallucination with high confidence: lists "hydroxyproline, hydroxychloroquine, hydroxychloroquine sulfate" as the active ingredients of ibuprofen (none of which are correct — the actual active ingredient is ibuprofen itself, an NSAID). Most dangerous failure mode in the comparison. |

### Part 3: Based on your evaluations, answer the following questions

**Q1. Overall, does RAG improve over no-RAG? Summarize your findings.**

Yes — substantially. Across all five query types, the RAG outputs are either correctly grounded in the retrieved passage or honestly refuse, while the no-RAG outputs hallucinate confidently in four out of five cases (the exception being the simple query, where no-RAG still fabricates a plausible-sounding but bogus method). The retrieved context functions as a hard prior that pulls the generator toward verifiable claims, while the no-RAG model relies on a 0.5B-parameter parametric memory that is too small to hold reliable technical knowledge and instead manufactures fluent nonsense.

**Q2. For which query types does RAG provide the most noticeable improvement, and why?**

The largest improvements are on the **knowledge-intensive** and **comparative** queries. These require specific facts (a precise flutter criterion; a specific tunnel-vs-tunnel comparison) that are exactly the kind of long-tail detail a small model cannot hold parametrically — so without retrieval the model invents them, but with retrieval it either cites the right detail (Comparative: "good agreement / compares favorably") or refuses honestly (Knowledge-intensive). The simple query, where the retrieved doc is only tangential, shows a smaller absolute improvement because both settings end up unhelpful — but RAG fails *honestly* (correct refusal) while no-RAG fails *confidently* (fluent fabrication).

**Q3. How much does retrieval affect hallucination? Does it consistently reduce hallucinations, or are there cases where it has little or no effect (or even introduces errors)? Discuss with examples.**

Retrieval reduces hallucination dramatically but does not eliminate it. The clearest reduction is on the comparative query: no-RAG inverts the answer ("not comparable") while RAG correctly states the agreement. The clearest *non-elimination* is the out-of-domain query: even with retrieval, the model fabricates a connection to the retrieved passage ("ibuprofen is included in the analysis for chemically active ablating surfaces"), turning a grounded refusal into a small hallucination grafted onto an unrelated document. So RAG's hallucination-reduction effect is conditional on retrieval *finding relevant content*; when retrieval returns irrelevant passages the model can still ground onto noise. This is a known failure mode of single-stage retrieval and motivates re-ranking or relevance gating.

**Q4. How does the overall quality of answers differ between RAG and no-RAG? Consider not just fluency but also substance and filler language.**

Both settings are roughly equally fluent surface-level, but the substance gap is large. RAG answers are short, document-anchored, and often quote fragments verbatim ("the induced pressure gradient is …", "compare favorably with previously published …"). No-RAG answers are longer, padded with formulaic intensifiers ("**these active ingredients work together to relieve pain by reducing inflammation and reducing pain**"), and frequently slip into degenerate paraphrase loops on the complex query. Filler language is the tell: when the model has nothing concrete to ground on, it compensates with repeated bolded phrases and tautologies — a signal that no-RAG is producing words rather than answers.

**Q5. How does the model behave on the out-of-domain query with and without RAG? Which behavior is more desirable from a user's perspective? Explain why.**

No-RAG fabricates the "active ingredients" of ibuprofen (hydroxyproline, hydroxychloroquine, hydroxychloroquine sulfate — none correct). RAG retrieves an unrelated Cranfield document on mass-transfer cooling and refuses, but with a stray hallucinated link ("ibuprofen is included in the analysis for chemically active ablating surfaces"). The RAG behavior is far more desirable: the user can read the cited document and immediately see it's irrelevant, so the refusal is auditable and recoverable. The no-RAG behavior is dangerous precisely because it sounds authoritative — a non-expert user has no signal that the listed compounds are wrong, and a wrong answer about a medication is exactly the kind of error a deployed system must avoid. Honest refusal beats confident fabrication.

## Task 4: Error Analysis

Analyse when the RAG pipeline succeeds or fails, and identify where failures originate — retrieval, generation, or both.


### Part 1: Identify One Example per Case

Using open-form queries, find **at least one example** of each case below. For each case, 

(a) determine whether the root cause is retrieval, generation, or both,

(b) propose a solution for the error cases.

| Case | Description | Diagnostic questions |
|------|-------------|----------------------|
| **Refusal** | The model declines to answer or expresses uncertainty | Was the context relevant and sufficient? Did the model fail to use available information? |
| **Incorrect Answer** | The model provides a wrong or unsupported answer | Was the context relevant? Did the model correctly use the retrieved information? |
| **Correct Answer** | The model provides a correct and relevant answer | Was the context relevant? Did the model correctly use the retrieved information? |


In [11]:
# Find queries that illustrate each case (Refusal / Incorrect / Correct).
# Same chosen model as Task 3 (model_completion via rag_system).

TASK4_PART1_QUERIES = [
    {"label": "Case A — refusal candidate (panel flutter criterion)",
     "query": "what is the criterion for true panel flutter, as opposed to small amplitude vibration arising from acoustic disturbances ."},
    {"label": "Case B — incorrect candidate (lift-drag ratios at high Mach)",
     "query": "what design factors can be used to control lift-drag ratios at mach numbers above 5 ."},
    {"label": "Case C — correct-answer candidate (structural / aeroelastic)",
     "query": "what are the structural and aeroelastic problems associated with flight of high speed aircraft ."},
    {"label": "Case D — out-of-domain (paracetamol)",
     "query": "What are the side effects of paracetamol overdose ?"},
]

K = 3
INSTR = "Answer the following technical question using only the information in the document."

for item in TASK4_PART1_QUERIES:
    print("=" * 90)
    print(item["label"])
    print(f"Query: {item['query']}")
    print()
    a, docs = rag_system.generate(item["query"], k=K, verbose=True, instruction=INSTR)
    print(f"Top retrieved (k={K}):")
    for i, d in enumerate(docs, 1):
        print(f"  [{i}] doc_id={d['doc_id']:>5}  score={d['score']:.2f}  title={d.get('title','').strip()[:70]}")
    print()
    print("--- Answer ---")
    print(a)
    print()


Case A — refusal candidate (panel flutter criterion)
Query: what is the criterion for true panel flutter, as opposed to small amplitude vibration arising from acoustic disturbances .



Top retrieved (k=3):
  [1] doc_id=  766  score=21.95  title=experimental investigation at mach number of 3. 0 of
effects of therma
  [2] doc_id=  827  score=19.45  title=a nonlinear theory of bending and buckling of thin
elastic shallow sph
  [3] doc_id=  627  score=19.25  title=flutter analysis of circular panels .

--- Answer ---
The provided document does not contain any information about the criterion for true panel flutter or the criterion for small-amplitude vibration arising from acoustic disturbances. It only states that flutter analysis of circular panels is formulated in terms of small-deflection plate theory and that the panel is subjected to uniform all-round tension or compression in its middle plane, with linear piston theory employed to predict the aerodynamic load on the vibrating panel. It does not mention any criterion for true flutter or acoustic disturbances flutter.

Case B — incorrect candidate (lift-drag ratios at high Mach)
Query: what design factors can be used

Top retrieved (k=3):
  [1] doc_id= 1188  score=28.23  title=factors affecting lift-drag ratios at mach numbers
from 5 to 20 .
  [2] doc_id= 1380  score=23.04  title=the problem of obtaining high lift-drag ratios at supersonic speeds .
  [3] doc_id=  674  score=18.77  title=the shapes and lift-dependent drags on some sweptback
wings designed f

--- Answer ---
The document does not mention any specific design factors that can be used to control lift–drag ratios at Mach numbers above 5. It only states that lift–drag ratios are calculated at selected flight design points for flat-top, conical body arrangements with triangular and hyperbolic wing plans, and that the lift–drag ratio is proportional to the lift-to-drag ratio, the thermopropulsive efficiency, and the initial to final weight ratio. It does not provide any specific design factors or how to control lift–drag ratios beyond that.

Case C — correct-answer candidate (structural / aeroelastic)
Query: what are the structural and aeroel

Top retrieved (k=3):
  [1] doc_id=   12  score=30.44  title=some structural and aerelastic considerations of high
speed flight .
  [2] doc_id=  746  score=20.60  title=aeroelastic problems in connection with high speed
flight .
  [3] doc_id=   51  score=18.00  title=theory of aircraft structural models subjected to aerodynamic
heating 

--- Answer ---
The document does not list any structural or aeroelastic problems associated with high-speed flight. It only discusses aeroelastic problems in connection with high-speed flight, including:

- A review of developments in aeroelasticity during the past ten years.
- Changes in aerodynamic derivatives, design for higher Mach numbers, and thinner aerofoils and more slender fuselages.
- Attacks and alleviations of structural and aeroelastic problems.
- Relative merits of stiffness, damping, and massbalance for prevention of control surface flutter.
- Mention of recent problems of damage from jet efflux and possible aeroelastic effects of kineti

Top retrieved (k=3):
  [1] doc_id=  210  score=9.53  title=propeller in yaw .
  [2] doc_id=  974  score=8.34  title=approximate analysis of thrust vector control by fluid injection .
  [3] doc_id= 1326  score=8.10  title=interaction of secondary injectants and rocket exhaust
for thrust vect

--- Answer ---
The document does not mention paracetamol overdose or any side effects associated with it. It discusses side effects of injection of secondary material into the nozzle, including side-force generation, nozzle-wall pressure profiles, and shock interfaces, but it does not address paracetamol overdose or its effects.



#### Case Analysis

**Refusal case — Case A: "what is the criterion for true panel flutter, as opposed to small amplitude vibration arising from acoustic disturbances ."**
- Query: as above (qid 191, held-out test set).
- Was the retrieved context relevant and sufficient? Partially. With k=3 the retriever surfaces **doc 627 ("flutter analysis of circular panels")** and **doc 766** (panel flutter under thermal stress) — both clearly on-topic. However, none of the retrieved abstracts explicitly state a numerical or analytical *criterion* that distinguishes true flutter from acoustic small-amplitude vibration; they describe the modeling framework (small-deflection plate theory, linear piston theory) but not the criterion itself.
- Did the model fail to use available information? Mostly no — the model correctly identifies that the criterion isn't stated and refuses with a faithful summary of what *is* in the document. This is the safe behavior, not a hallucination.
- Root cause: **retrieval (primary) + generation (secondary)**. The corpus contains the relevant abstracts but at abstract-level granularity the criterion language doesn't appear; the model could not have lifted it from the retrieved snippets. A more conservative diagnosis is that this is an honest refusal driven by the limits of what abstracts can carry.
- Proposed solution: (i) index full-text or longer chunks rather than the title+abstract pairs Cranfield ships, so the actual flutter-criterion sentences become retrievable; (ii) raise k and apply a re-ranker to surface the abstract that most closely defines a "criterion"; (iii) prompt-side, allow synthesis with explicit citation when several abstracts independently describe the same phenomenon.

---

**Incorrect answer case — Case B: "what design factors can be used to control lift-drag ratios at mach numbers above 5 ."**
- Query: as above.
- Was the retrieved context relevant and sufficient? **Yes — the top hit (doc 1188, score 28.2) is literally titled "factors affecting lift-drag ratios at mach numbers from 5 to 20"**. This is essentially a perfect retrieval — the title answers the question.
- Did the model correctly use the retrieved information? **No.** The model says "The document does not mention any specific design factors that can be used to control lift–drag ratios at Mach numbers above 5", then paraphrases unrelated calculation parameters (thermopropulsive efficiency, weight ratios). This is a clear case where the retrieved context contains the answer but the generator fails to recognize it.
- Root cause: **generation**. Retrieval did its job; the generator either does not parse the title+abstract as containing "design factors" (a wording mismatch — the doc uses "factors affecting" rather than "design factors") or has a learned prior toward refusal when the question phrasing doesn't appear verbatim in the passage.
- Proposed solution: (i) at the prompt level, instruct the model to extract design factors even if they appear under different phrasings; (ii) post-process by re-reading the top doc and asking the model "list every factor mentioned in the document that affects lift-drag ratio"; (iii) improve the fine-tuning data with examples where the question wording differs from the document wording so the model learns to bridge synonyms.

---

**Correct answer case — Case C: "what are the structural and aeroelastic problems associated with flight of high speed aircraft ."**
- Query: as above.
- Was the retrieved context relevant and sufficient? **Yes** — top-3 retrieval brings doc 12 ("some structural and aeroelastic considerations of high speed flight"), doc 746 ("aeroelastic problems in connection with high speed flight"), and doc 51 (theory of aircraft structural models with aerodynamic heating). All directly on-topic and high-scoring (30.4, 20.6, 18.0).
- Did the model correctly use the retrieved information? **Yes.** Despite an awkward "the document does not list..." opening, the model then proceeds to enumerate the actual problems lifted from doc 746: changes in aerodynamic derivatives, design for higher Mach numbers, thinner airfoils and more slender fuselages, control-surface-flutter prevention (stiffness vs. damping vs. mass-balance), jet-efflux damage, and kinetic-heating aeroelastic effects. These are real document quotes.
- Root cause: success — retrieval brought directly relevant abstracts and the generator extracted concrete content from them. The vestigial "does not list" prefix is a learned hedge from the SFT data and is harmless here since the actual list follows.

### Part 2: Experiment with at least two additional values of $k$

Select one query where retrieval was the cause of failure and one where the generator was the cause. 

Experiment at least two additional values of $k$ for each (one higher and one lower than your original setting) and discuss the results. 



In [12]:
# Pick one query where retrieval was the cause of failure, and one where the
# generator was the cause. Sweep k = 1, 3, 5 for each and inspect outputs.
#
#   Retrieval failure : the out-of-domain paracetamol query (no relevant doc
#                        exists in the Cranfield corpus, so no value of k can
#                        retrieve relevant content).
#   Generator failure : the lift-drag-ratio query — doc 1188 is a near-perfect
#                        retrieval, yet the model refused.

K_SWEEP = [
    {"label": "Retrieval failure (out-of-domain)",
     "query": "What are the side effects of paracetamol overdose ?",
     "ks":    [1, 3, 5]},
    {"label": "Generator failure (high-Mach lift-drag)",
     "query": "what design factors can be used to control lift-drag ratios at mach numbers above 5 .",
     "ks":    [1, 3, 5]},
]

INSTR = "Answer the following technical question using only the information in the document."

for item in K_SWEEP:
    print("=" * 90)
    print(item["label"])
    print(f"Query: {item['query']}")
    print()
    for k in item["ks"]:
        print(f"--- k = {k} ---")
        a, docs = rag_system.generate(item["query"], k=k, verbose=True, instruction=INSTR)
        for i, d in enumerate(docs, 1):
            print(f"  [{i}] doc_id={d['doc_id']:>5}  score={d['score']:.2f}  title={d.get('title','').strip()[:70]}")
        print(f"  Answer: {a}")
        print()


Retrieval failure (out-of-domain)
Query: What are the side effects of paracetamol overdose ?

--- k = 1 ---


  [1] doc_id=  210  score=9.53  title=propeller in yaw .
  Answer: The document does not mention paracetamol overdose or any side effects associated with it. It only states that propeller forces due to an angular velocity of pitch are “very small” for pitching velocities that may actually be realized in maneuvers, with the exception of spin, and that yawed single-rotating propellers experience a pitching moment in addition to the side force. It also notes that the formulas for propellers in yaw are derived without derivation in an earlier report.

--- k = 3 ---


  [1] doc_id=  210  score=9.53  title=propeller in yaw .
  [2] doc_id=  974  score=8.34  title=approximate analysis of thrust vector control by fluid injection .
  [3] doc_id= 1326  score=8.10  title=interaction of secondary injectants and rocket exhaust
for thrust vect
  Answer: The document does not mention paracetamol overdose or any side effects associated with it. It discusses side effects of injection of secondary material into the nozzle, including side-force generation, nozzle-wall pressure profiles, and shock interfaces, but it does not address paracetamol overdose or its effects.

--- k = 5 ---


  [1] doc_id=  210  score=9.53  title=propeller in yaw .
  [2] doc_id=  974  score=8.34  title=approximate analysis of thrust vector control by fluid injection .
  [3] doc_id= 1326  score=8.10  title=interaction of secondary injectants and rocket exhaust
for thrust vect
  [4] doc_id= 1167  score=7.81  title=an experimental study of the effect of downwash from
a twin-propeller 
  [5] doc_id= 1251  score=7.29  title=viscous flow past a quarter infinite plate .
  Answer: The provided document does not contain any information about propeller-side effects of paracetamol overdose. It only discusses viscous flow past a quarter-infinite plate and notes that the viscous flow is “considered separately,” with the leading edge being “perpendicular to and parallel to the undisturbed direction of the stream,” and that the shear layer “leads to a velocity component normal to the plate” that “needs a potential solution,” with the velocities matching with infinity. It does not mention paracetamol overd

  [1] doc_id= 1188  score=28.23  title=factors affecting lift-drag ratios at mach numbers
from 5 to 20 .
  Answer: The document does not mention any specific design factors that can be used to control lift–drag ratios at Mach numbers above 5. It only states that lift–drag ratios can be calculated at selected flight design points for **flat-top, conical body arrangements with triangular and hyperbolic wing plansforms** and that **hyperbolic wing arrangement offers a potential L/D benefit at Mach 5** but not at **Mach 10 or above**. It does not provide any specific design factors for Mach 5 or beyond.

--- k = 3 ---


  [1] doc_id= 1188  score=28.23  title=factors affecting lift-drag ratios at mach numbers
from 5 to 20 .
  [2] doc_id= 1380  score=23.04  title=the problem of obtaining high lift-drag ratios at supersonic speeds .
  [3] doc_id=  674  score=18.77  title=the shapes and lift-dependent drags on some sweptback
wings designed f
  Answer: The document does not mention any specific design factors that can be used to control lift–drag ratios at Mach numbers above 5. It only states that lift–drag ratios are calculated at selected flight design points for flat-top, conical body arrangements with triangular and hyperbolic wing plans, and that the lift–drag ratio is proportional to the lift-to-drag ratio, the thermopropulsive efficiency, and the initial to final weight ratio. It does not provide any specific design factors or how to control lift–drag ratios beyond that.

--- k = 5 ---


  [1] doc_id= 1188  score=28.23  title=factors affecting lift-drag ratios at mach numbers
from 5 to 20 .
  [2] doc_id= 1380  score=23.04  title=the problem of obtaining high lift-drag ratios at supersonic speeds .
  [3] doc_id=  674  score=18.77  title=the shapes and lift-dependent drags on some sweptback
wings designed f
  [4] doc_id=  225  score=18.71  title=elliptic cones alone and with wings at supersonic speeds .
  [5] doc_id=  638  score=18.01  title=longitudinal aerodynamic characteristics at low subsonic
speeds of a h
  Answer: The document does not list specific design factors that can be used to control lift–drag ratios at Mach numbers above 5. It only states that **“use of nose control”** can improve control effectiveness for high control deflections, and that **“deflection of the lower-surface split flap”** can reduce trimmed angle of attack for a given value of trimmed lift coefficient. It does not provide any other design factors beyond that.



#### Discussion

**Does increasing or decreasing $k$ resolve the error, introduce new ones, or make no difference? What does this suggest about the advantages and limitations of retrieval depth?**

The two cases give opposite answers and together expose what k can and cannot fix.

For the **retrieval-failure** query (out-of-domain paracetamol), changing k has **no effect on correctness**: at k=1, 3, and 5 the retrieved docs are aerospace papers (propellers in yaw, thrust vector control, viscous flow past a plate) and the model correctly refuses each time. The only thing that changes is *which* irrelevant doc appears in the refusal text — at k=5 the model latches onto "viscous flow past a quarter-infinite plate" as the doc-anchored detail. This is the fundamental limitation of retrieval depth: when the corpus simply does not contain the answer, no value of k can fix that — every retrieval is noise. Increasing k here only makes the refusal slightly more confused because the model has more competing irrelevant signals to ground onto.

For the **generator-failure** query (lift-drag ratios at Mach > 5), k changes the *flavour* of the wrong answer but does not fix it. At k=1, the model surfaces some real document content from doc 1188 ("hyperbolic wing offers a potential L/D benefit at Mach 5 but not at Mach 10 or above") — a genuine extracted fact, even if hedged. At k=3, the additional docs dilute the signal: the model now mentions thermopropulsive efficiency and weight ratios (from a different doc) and produces a less specific answer. At k=5, the answer drifts further, surfacing "nose control" and "lower-surface split flap" — which are real design factors but from yet another document, while still refusing to call them an answer. So increasing k *introduces new content* (sometimes relevant) but also introduces *context dilution*: the highest-scoring, most-on-topic doc loses prompt-share, and the model's already-shaky signal for "this is the design-factor doc" gets weaker.

The takeaway: **k is a tuning knob on the noise/signal trade-off, not a fix for orthogonal failure modes.** Larger k expands the chance of including relevant content (helpful when the top-1 was not enough), but at the cost of more competing irrelevant context (harmful when retrieval was already on-target). For a small generator like the 0.5B Qwen used here, the dilution effect is especially harsh — its limited attention budget means each extra document costs grounding precision on the truly relevant one. The right response to a retrieval miss is *better retrieval* (re-ranking, query rewriting, better chunking) or *more corpus*, not larger k. The right response to a generator miss is *prompt engineering* or *better training data* (paraphrase coverage, anti-refusal training), again not larger k.